In [ ]:
!pip install dotenv

In [ ]:
!pip install langchain pymupdf faiss-cpu google-generativeai langchain-community

In [1]:
from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings.base import Embeddings
from typing import List
import google.generativeai as genai
import asyncio

d:\Ky 4\vietnamese-speech-chatbox\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
import os

secret_key = os.getenv("SECRET_KEY")
genai.configure(api_key=secret_key)

In [3]:
models = genai.list_models()
for model in models:
    print(model.name)

models/embedding-gecko-001
models/gemini-1.0-pro-vision-latest
models/gemini-pro-vision
models/gemini-1.5-pro-latest
models/gemini-1.5-pro-001
models/gemini-1.5-pro-002
models/gemini-1.5-pro
models/gemini-1.5-flash-latest
models/gemini-1.5-flash-001
models/gemini-1.5-flash-001-tuning
models/gemini-1.5-flash
models/gemini-1.5-flash-002
models/gemini-1.5-flash-8b
models/gemini-1.5-flash-8b-001
models/gemini-1.5-flash-8b-latest
models/gemini-1.5-flash-8b-exp-0827
models/gemini-1.5-flash-8b-exp-0924
models/gemini-2.5-pro-exp-03-25
models/gemini-2.5-pro-preview-03-25
models/gemini-2.5-flash-preview-04-17
models/gemini-2.5-flash-preview-05-20
models/gemini-2.5-flash-preview-04-17-thinking
models/gemini-2.5-pro-preview-05-06
models/gemini-2.5-pro-preview-06-05
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-preview-image-generation


In [4]:
pdf_path = r"C:\Users\5530\Downloads\att.pdf"
loader = PyMuPDFLoader(pdf_path)
documents = loader.load()

In [5]:
text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
)
chunks = text_splitter.split_documents(documents)

for chunk in chunks:
    print(chunk.page_content)
    break

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Exper

In [3]:
import pandas as pd
from langchain.schema import Document

# Đường dẫn đến file CSV
csv_path = r"D:\Ky 4\vietnamese-speech-chatbox\notebooks\Crawler\DataCollection\syllabus_data.csv"

# Đọc file CSV
df = pd.read_csv(csv_path)

# Tạo các chunks: mỗi dòng trong CSV là một chunk
chunks = []
for i, row in df.iterrows():
    # Chuyển từng dòng thành một Document object (nếu cần cho LangChain)
    text = ' '.join(str(cell) for cell in row.values)
    chunks.append(Document(page_content=text))

# In thử một chunk
print(chunks[0].page_content)

26583c76-d115-cd39-24a4-49fb319885a1 https://flm.fpt.edu.vn/gui/role/student/SyllabusDetails?sylid=11845 ID giáo trình:: 11845
Tên giáo trình:: Communication and In-Group Working Skills_Kỹ năng giao tiếp và cộng tác
Giáo trình tiếng Anh:: 
Mã chủ đề:: SSG104
Số tín chỉ:: 3
Cấp độ:: Cử nhân
Phân bổ thời gian:: Giờ học (150h) = 45 giờ liên lạc (60 phiên) + kỳ thi cuối cùng 0,5 giờ + 104,5 giờ tự học
Điều kiện tiên quyết:: Không có
Sự miêu tả:: Khóa học này bao gồm cả làm việc trong nhóm và kỹ năng giao tiếp. Khóa học bao gồm các lý thuyết về giao tiếp, làm việc theo nhóm và các hoạt động để sinh viên thực hành áp dụng các lý thuyết trong bối cảnh học tập và làm việc.
Nhiệm vụ sinh viên:: - Học sinh phải tham dự ít nhất 80% các khe liên hệ để được chấp nhận vào kỳ thi cuối cùng. - Học sinh có trách nhiệm thực hiện tất cả các bài tập và bài tập do người hướng dẫn trong lớp hoặc ở nhà được đưa ra và nộp đúng hạn - chỉ sử dụng máy tính xách tay trong lớp cho mục đích học tập
Công cụ:: - Inte

In [4]:
class GeminiEmbeddings(Embeddings):
    def __init__(self, model_name="models/text-embedding-004"):
        self.model_name = model_name

    def embed_query(self, query: str) -> List[float]:
        try:
            result = genai.embed_content(
                model=self.model_name,
                content=query
            )
            return result['embedding']
        except Exception as e:
            print(f"Error embedding query: {e}")
            return []

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        embeddings = []
        for text in texts:
            try:
                result = genai.embed_content(
                    model=self.model_name,
                    content=text
                )
                embeddings.append(result['embedding'])
            except Exception as e:
                print(f"Error embedding document: {e}")
                embeddings.append([])
        return embeddings

embeddings = GeminiEmbeddings()

In [5]:
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")

In [6]:
vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

In [10]:
class SmartChatbot:
    def __init__(self, vectorstore, model_name="models/gemini-2.0-flash-exp"):
        self.vectorstore = vectorstore
        self.model_name = model_name

    async def answer_question(self, question: str) -> str:
        try:
            # Retrieve relevant documents using the vectorstore
            related_docs = self.vectorstore.similarity_search(question, k=30)
            context = "\n\n".join([doc.page_content for doc in related_docs])

            # Generate a response using the LLM with context
            prompt = (
                f"You are a helpful assistant. Use the following context to answer:\n\n"
                f"{context}\n\n"
                f"Question: {question}\n"
                f"Answer:"
            )

            # Use GenerativeModel from google.generativeai
            model = genai.GenerativeModel(self.model_name)
            response = model.generate_content(prompt)
            return response.text
        except Exception as e:
            return f"Error: {e}"


async def simulate_typing(text):
    for char in text:
        print(char, end='', flush=True)
        await asyncio.sleep(0.0002)
    print()

async def main():
    # Initialize the chatbot
    chatbot = SmartChatbot(vectorstore)

    # Ask a question
    question = "Tôi muốn biết thêm về môn học thị giác máy tính. Bạn có thể cung cấp thông tin chi tiết không?"

    print(f"\nYou: {question}")
    print("\nBot is typing...", end='\n')
    await asyncio.sleep(1)
    response = await chatbot.answer_question(question)
    print(" " * 15, end='\r')  # Clear "Bot is typing..."
    print()
    await simulate_typing(response)

await main()


You: Tôi muốn biết thêm về môn học thị giác máy tính. Bạn có thể cung cấp thông tin chi tiết không?

Bot is typing...
               
Chắc chắn rồi! Dưới đây là thông tin chi tiết về môn học Thị giác máy tính (Computer Vision):

**ID giáo trình:** 8972
**Tên giáo trình:** Computer Vision (Thị giác máy tính)
**Giáo trình tiếng Anh:**
**Mã chủ đề:** CPV301
**Số tín chỉ:** 3
**Cấp độ:** Cử nhân
**Phân bổ thời gian:** 45 giờ học (60 phiên) + 1 giờ thi cuối kỳ + 104 giờ tự học
**Điều kiện tiên quyết:** PFP191, CSD203
**Mô tả:**

*   Khóa học này cung cấp những kiến thức cơ bản về thị giác máy tính.
*   Sinh viên sẽ được tiếp cận kiến thức về biểu diễn hình ảnh, ánh sáng, thu thập hình ảnh thông qua máy ảnh, kỹ thuật hiệu chỉnh camera.
*   Học các kỹ thuật phát hiện đường thẳng (Line) và cạnh (edge) trong hình ảnh, bộ lọc (filters) cũng như Canny, các phương thức RANSAC.
*   Tìm hiểu kiến thức xử lý hình ảnh nâng cao như phân đoạn hình ảnh, phát hiện đối tượng, nhận dạng đối tượng và theo d

# https://ai.google.dev/gemini-api/docs/rate-limits

# https://ai.google.dev/gemini-api/docs/pricing

🔹 1. Câu hỏi kiến thức tổng quát
Ai là người viết bản Tuyên ngôn Độc lập của Việt Nam?

Thủ đô của Nhật Bản là gì?

Trái đất quay quanh mặt trời mất bao lâu?

Năm 1975 có sự kiện gì quan trọng ở Việt Nam?

Sông nào dài nhất Việt Nam?

🔹 2. Câu hỏi logic – suy luận
Nếu hôm nay là thứ Ba, thì ba ngày nữa là thứ mấy?

Tôi có 5 quả táo, ăn mất 2 quả. Hỏi còn lại bao nhiêu quả?

Nếu A lớn hơn B, và B lớn hơn C, thì A có lớn hơn C không?

🔹 3. Câu hỏi tương tác tự nhiên
Hãy kể một câu chuyện cười ngắn.

Gợi ý giúp tôi một món ăn tối nhanh và dễ làm.

Cho tôi vài lời khuyên để học tiếng Anh hiệu quả.

Tôi đang buồn, bạn có thể an ủi tôi được không?

🔹 4. Câu hỏi sáng tạo / viết lách
Viết một đoạn thơ ngắn về mùa xuân.

Hãy viết một bức thư xin lỗi bạn thân.

Hãy mô tả một con rồng trong tưởng tượng của bạn.

🔹 5. Câu hỏi chuyên sâu (tùy mục tiêu chatbot)
Giải thích ngắn gọn về mạng nơ-ron nhân tạo.

Tôi muốn học lập trình Python, bắt đầu từ đâu?

GPT khác BERT như thế nào?

🔹 6. Câu hỏi kiểm tra giới hạn mô hình
Hãy tóm tắt bài thơ “Truyện Kiều”.



In [1]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
print("CUDA khả dụng:", torch.cuda.is_available())
print("Tên GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Không có GPU")
# Đảm bảo bạn đã đặt biến môi trường HUGGINGFACE_TOKEN
HF_TOKEN = os.getenv("HUGGINGFACE_TOKEN")  # hoặc thay bằng chuỗi token nếu bạn test
os.environ["TRANSFORMERS_NO_TF"] = "1"  # tránh xung đột với TensorFlow

# Cấu hình để tải model 4-bit
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

# Load tokenizer và model Gemma 2B (Google)
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b", token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b",
    quantization_config=quantization_config,
    device_map="auto",
    token=HF_TOKEN
).to("cuda")

# Prompt đầu vào
prompt = "Thủ đô của Nhật Bản là gì?"

# Tokenize đầu vào
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Sinh văn bản
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True
    )

# Giải mã và in kết quả
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n🧠 Model trả lời:\n", response)


d:\Ky 4\vietnamese-speech-chatbox\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA khả dụng: True
Tên GPU: Quadro P2000


Loading checkpoint shards: 100%|██████████| 2/2 [00:13<00:00,  6.55s/it]
d:\Ky 4\vietnamese-speech-chatbox\.venv\lib\site-packages\bitsandbytes\nn\modules.py:451: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(



🧠 Model trả lời:
 Thủ đô của Nhật Bản là gì?

Nêu thành tựu văn hoá nổi bật của các quốc gia: La-10, Xi-ri-oa, Pa-li- Motosơ.

Ai là chủ tịch 12 nước châu âu trong giai đoạn 1961-1973

A.các quốc gia Ả- 13

B.các quốc gia Ả - 5

Đại hội thống nhất Đông Dương họp lần 1 tại đâu?

<strong>1.</strong>


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Load model (code bạn đã có)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True
)

print("Đang tải tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("Viet-Mistral/Vistral-7B-Chat")

print("Đang tải model...")
model = AutoModelForCausalLM.from_pretrained(
    "Viet-Mistral/Vistral-7B-Chat",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model đã sẵn sàng!\n")

def chat_with_bot(user_input, max_new_tokens=512):
    """
    Hàm để chat với Vistral bot
    """
    # Format prompt theo style của Vistral
    prompt = f"<|im_start|>user\n{user_input}<|im_end|>\n<|im_start|>assistant\n"
    
    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)
    
    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            top_k=40,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # Decode response
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract only the assistant's response
    response = full_response.split("<|im_start|>assistant\n")[-1].strip()
    
    return response

def interactive_chat():
    """
    Chế độ chat tương tác
    """
    print("=== VISTRAL CHATBOT ===")
    print("Gõ 'quit' hoặc 'exit' để thoát\n")
    
    while True:
        user_input = input("Bạn: ")
        
        if user_input.lower() in ['quit', 'exit', 'thoát']:
            print("Tạm biệt!")
            break
        
        if user_input.strip() == "":
            continue
            
        print("Bot đang trả lời...")
        response = chat_with_bot(user_input)
        print(f"Vistral: {response}\n")

# Các câu hỏi mẫu để test
def test_chatbot():
    """
    Test chatbot với một số câu hỏi mẫu
    """
    test_questions = [
        "Xin chào, bạn có thể giới thiệu về bản thân không?",
        "Hà Nội là thủ đô của nước nào?",
        "Viết một bài thơ ngắn về mùa thu",
        "Giải thích về trí tuệ nhân tạo bằng tiếng Việt"
    ]
    
    print("=== TEST CHATBOT ===\n")
    
    for i, question in enumerate(test_questions, 1):
        print(f"Câu hỏi {i}: {question}")
        response = chat_with_bot(question)
        print(f"Trả lời: {response}\n")
        print("-" * 50 + "\n")

if __name__ == "__main__":
    # Chọn chế độ
    print("Chọn chế độ:")
    print("1. Chat tương tác")
    print("2. Test với câu hỏi mẫu")
    
    choice = input("Nhập lựa chọn (1 hoặc 2): ").strip()
    
    if choice == "1":
        interactive_chat()
    elif choice == "2":
        test_chatbot()
    else:
        print("Lựa chọn không hợp lệ!")
        
        # Hoặc bạn có thể trực tiếp hỏi như này:
        question = "Xin chào, bạn là ai?"
        print(f"Câu hỏi: {question}")
        answer = chat_with_bot(question)
        print(f"Trả lời: {answer}")

Đang tải tokenizer...
Đang tải model...


Loading checkpoint shards: 100%|██████████| 2/2 [00:27<00:00, 13.80s/it]
Some parameters are on the meta device because they were offloaded to the cpu.


Model đã sẵn sàng!

Chọn chế độ:
1. Chat tương tác
2. Test với câu hỏi mẫu
